# Fine-Tuned Neural Text Classification Model

We train a neural text classification model using DistilBERT.

The goal is to compare a transformer-based model with the classical TF-IDF + Logistic Regression baseline.  
Because transformer models require more computing power, we use a smaller balanced sample from the dataset for training and validation.

In [1]:
# Import basic libraries
import pandas as pd
import numpy as np
import os
import torch

# Import evaluation tools
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Import Hugging Face transformer tools
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check whether GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

Using device: cpu


In [3]:
# Load the data splits created in Notebook 2
train_df = pd.read_csv("../Data/train_tweets.csv")
val_df = pd.read_csv("../Data/val_tweets.csv")
test_df = pd.read_csv("../Data/test_tweets.csv")

# dataset sizes
print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

train_df.head()

Training set: (41428, 4)
Validation set: (8878, 4)
Test set: (8878, 4)


,tweet_id,text,clean_text,company
0,2955447,@AppleSupport Trying to set up a new iPhone bu...,trying to set up a new iphone but keep getting...,applesupport
1,941263,"Hey, @SouthwestAir, I think you guys are prett...",hey i think you guys are pretty great too thx,southwestair
2,1635741,@AppleSupport iPhone 7plus and the latest iOS,iphone plus and the latest ios,applesupport
3,485214,"@AmazonHelp Spoke with a nice lil lady, but al...",spoke with a nice lil lady but alas i see from...,amazonhelp
4,1297186,@Uber_Support Ok.. Thank you I will send you d...,ok thank you i will send you details complaint...,uber_support


In [5]:
# Function to sample a fixed number of examples per company
def sample_per_class(data, label_col, n_per_class, random_state=42):
    sampled_parts = []
    
    # Loop through each company group
    for label in sorted(data[label_col].unique()):
        group = data[data[label_col] == label]
        
        # Sample up to n_per_class rows from this company
        sampled_group = group.sample(
            n=min(len(group), n_per_class),
            random_state=random_state
        )
        
        sampled_parts.append(sampled_group)
    
    # Combine all sampled company groups
    sampled_data = pd.concat(sampled_parts, axis=0).reset_index(drop=True)
    
    return sampled_data

# Create smaller balanced datasets for neural model training
train_small = sample_per_class(train_df, "company", n_per_class=500)
val_small = sample_per_class(val_df, "company", n_per_class=150)
test_small = sample_per_class(test_df, "company", n_per_class=150)

print("Small training set:", train_small.shape)
print("Small validation set:", val_small.shape)
print("Small test set:", test_small.shape)

print("\ntrain_small columns:", train_small.columns.tolist())

Small training set: (4500, 4)
Small validation set: (1350, 4)
Small test set: (1350, 4)

train_small columns: ['tweet_id', 'text', 'clean_text', 'company']


To make fine-tuning computationally manageable, we use a smaller balanced sample from each company class.  
This also reduces the effect of class imbalance during neural model training.

In [6]:
# Convert company names into numeric labels
label_encoder = LabelEncoder()

# Fit label encoder on the training labels
train_small["label"] = label_encoder.fit_transform(train_small["company"])

# using encoder on validation and test labels
val_small["label"] = label_encoder.transform(val_small["company"])
test_small["label"] = label_encoder.transform(test_small["company"])

# Store number of classes
num_labels = len(label_encoder.classes_)

print("Number of labels:", num_labels)
print("Labels:", list(label_encoder.classes_))

Number of labels: 9
Labels: ['amazonhelp', 'americanair', 'applesupport', 'delta', 'southwestair', 'spotifycares', 'tesco', 'uber_support', 'virgintrains']


## Fine-Tuning DistilBERT

We fine-tune DistilBERT for tweet company classification.  
Since training is done on CPU, we use a smaller balanced dataset and a limited number of epochs.

In [7]:
# Function used by the Trainer to calculate evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Convert model outputs into predicted class labels
    predictions = np.argmax(logits, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    weighted_f1 = f1_score(labels, predictions, average="weighted")
    
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [8]:
# Define the DistilBERT model name
model_name = "distilbert-base-uncased"

# Load the tokenizer again to make sure it exists
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Model name:", model_name)
print("Tokenizer loaded successfully.")

Model name: distilbert-base-uncased
Tokenizer loaded successfully.


In [14]:
# Tokenize training text
train_encodings = tokenizer(
    train_small["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# Tokenize validation text
val_encodings = tokenizer(
    val_small["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# Tokenize test text
test_encodings = tokenizer(
    test_small["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed.")

Tokenization completed.


In [15]:
# Create a PyTorch dataset class for transformer training
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert tokenized inputs into tensors
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        # Add the correct label
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


# Create datasets for training, validation, and testing
train_dataset = TweetDataset(
    train_encodings,
    train_small["label"].tolist()
)

val_dataset = TweetDataset(
    val_encodings,
    val_small["label"].tolist()
)

test_dataset = TweetDataset(
    test_encodings,
    test_small["label"].tolist()
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 4500
Validation dataset size: 1350
Test dataset size: 1350


In [16]:
# Load DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

print("DistilBERT model loaded successfully.")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4960.33it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully.


In [17]:
# Check that all required objects exist before training
print("Number of labels:", num_labels)

print("train_small shape:", train_small.shape)
print("val_small shape:", val_small.shape)
print("test_small shape:", test_small.shape)

print("train_dataset exists:", "train_dataset" in globals())
print("val_dataset exists:", "val_dataset" in globals())
print("test_dataset exists:", "test_dataset" in globals())

Number of labels: 9
train_small shape: (4500, 5)
val_small shape: (1350, 5)
test_small shape: (1350, 5)
train_dataset exists: True
val_dataset exists: True
test_dataset exists: True


In [18]:
# Save model outputs
os.makedirs("../results/models/distilbert_baseline", exist_ok=True)

# Training settings for the baseline DistilBERT model
training_args = TrainingArguments(
    output_dir="../results/models/distilbert_baseline",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

print("Training arguments created successfully.")

Training arguments created successfully.


In [19]:
# Create Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

Trainer created successfully.


In [20]:
# Train the DistilBERT baseline model
trainer.train()

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.373067,1.230888,0.522222,0.500742,0.500742


TrainOutput(global_step=563, training_loss=1.4687673219779034, metrics={'train_runtime': 1441.9264, 'train_samples_per_second': 3.121, 'train_steps_per_second': 0.39, 'total_flos': 111783320352000.0, 'train_loss': 1.4687673219779034, 'epoch': 1.0})

In [21]:
# Evaluate the trained model on the validation set
bert_val_results = trainer.evaluate()

print("DistilBERT validation results:")
print(bert_val_results)

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
1.373067,1.230888,1,0.522222,0.500742,0.500742


DistilBERT validation results:
{'eval_loss': 1.2308884859085083, 'eval_accuracy': 0.5222222222222223, 'eval_macro_f1': 0.5007419344603814, 'eval_weighted_f1': 0.5007419344603815}
